## 1. Introduzione a Serverless

- **Serverless** non significa "senza server", ma *senza gestire i server*: il provider (AWS) si occupa di provisioning, scaling e manutenzione.
- Si paga **solo quando il codice viene eseguito** (pay-per-use); a riposo il costo è zero.
- Scaling automatico: da 0 a molte richieste parallele senza configurazione.
- Servizio AWS di riferimento: **AWS Lambda** (Function-as-a-Service). Il free tier include 1 milione di richieste e 400.000 GB-secondi al mese.
- **Caso d'uso del modulo**: deployare il modello di classificazione di vestiti (capitolo 8, Keras/Xception) come servizio:
  `URL immagine -> Lambda (modello) -> predizione della classe`
- Flusso generale del modulo:
  1. Convertire il modello Keras in **TF Lite**
  2. Scrivere la `lambda_function.py`
  3. Creare un'**immagine Docker** per Lambda
  4. Pubblicarla su **ECR** e creare la funzione Lambda
  5. Esporre la funzione con **API Gateway**
- Vantaggi: zero gestione infrastruttura, scaling automatico, costi bassi per carichi variabili.
- Limiti: *cold start*, limiti di memoria/timeout/dimensione del pacchetto, meno adatto a carichi costanti e molto alti.

## 2. AWS Lambda

- Una Lambda è una funzione con un **handler** che riceve due argomenti:
  - `event`: i dati in ingresso (dizionario, di solito JSON)
  - `context`: informazioni sull'esecuzione (spesso non usato)
- Il valore restituito deve essere serializzabile in JSON.
- Si può creare e testare dalla console AWS (runtime Python), definendo un *test event*.

In [ ]:
# lambda_function.py – esempio minimo
def lambda_handler(event, context):
    print("parameters:", event)
    url = event["url"]
    return {"message": f"ricevuto {url}"}

# Test event (JSON) da incollare nella console AWS:
# {"url": "http://example.com/image.jpg"}

- Il nome dell'handler nella configurazione è `<nome_file>.<nome_funzione>` (es. `lambda_function.lambda_handler`).
- Per la gestione tramite Docker si userà un'immagine base fornita da AWS (vedi capitolo 5).

## 3. TensorFlow Lite

- **TensorFlow completo è troppo pesante** per Lambda (dimensione del pacchetto, cold start lenti).
- **TF Lite** è pensato per la sola **inferenza**: più leggero e veloce, ideale per dispositivi con poche risorse e per Lambda.
- Si usa il pacchetto `tflite_runtime` (solo interprete) al posto di `tensorflow`.
- **Conversione Keras → TF Lite**:

In [ ]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.load_model("xception_v4_large_08_0.894.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("clothing-model.tflite", "wb") as f_out:
    f_out.write(tflite_model)

**Uso del modello TF Lite** (interprete):

In [ ]:
import tflite_runtime.interpreter as tflite   # in alternativa: tf.lite.Interpreter

interpreter = tflite.Interpreter(model_path="clothing-model.tflite")
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

# X: array numpy float32 con shape (1, 299, 299, 3), già preprocessato
interpreter.set_tensor(input_index, X)
interpreter.invoke()
preds = interpreter.get_tensor(output_index)

**Preprocessing senza Keras** (per evitare di installare TensorFlow):
- Scaricare l'immagine con `urllib` e aprirla con `PIL`
- Convertire in RGB, ridimensionare a **299×299**
- Convertire in array `float32` e applicare la normalizzazione di Xception: `x / 127.5 - 1`
- Aggiungere la dimensione batch

In [ ]:
from io import BytesIO
from urllib import request
from PIL import Image
import numpy as np

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    return Image.open(BytesIO(buffer))

def prepare_image(img, target_size=(299, 299)):
    if img.mode != "RGB":
        img = img.convert("RGB")
    return img.resize(target_size, Image.NEAREST)

def preprocess_input(x):
    x = x.astype("float32")
    x /= 127.5
    x -= 1.0
    return x

img = prepare_image(download_image("http://bit.ly/mlbookcamp-pants"))
X = preprocess_input(np.array(img)[None, ...])

## 4. Preparare il codice per Lambda

Passaggi:
1. Convertire il notebook in script: `jupyter nbconvert --to script notebook.ipynb`
2. Rinominare lo script in `lambda_function.py` e **ripulirlo** (solo ciò che serve).
3. Caricare modello e interprete **fuori dall'handler**, così vengono inizializzati una sola volta (riutilizzati nelle invocazioni "calde").
4. Definire `predict(url)` e `lambda_handler(event, context)`.
5. Testare **in locale** prima del deploy.

In [ ]:
# lambda_function.py
import tflite_runtime.interpreter as tflite
from io import BytesIO
from urllib import request
from PIL import Image
import numpy as np

interpreter = tflite.Interpreter(model_path="clothing-model.tflite")
interpreter.allocate_tensors()
input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

classes = ["dress", "hat", "longsleeve", "outwear", "pants",
           "shirt", "shoes", "shorts", "skirt", "t-shirt"]

# ... funzioni download_image, prepare_image, preprocess_input (vedi sopra) ...

def predict(url):
    img = prepare_image(download_image(url))
    X = preprocess_input(np.array(img)[None, ...])
    interpreter.set_tensor(input_index, X)
    interpreter.invoke()
    preds = interpreter.get_tensor(output_index)
    return dict(zip(classes, preds[0].tolist()))   # tolist(): JSON serializzabile

def lambda_handler(event, context):
    url = event["url"]
    return predict(url)

In [ ]:
# Test locale in Python
import lambda_function
lambda_function.lambda_handler({"url": "http://bit.ly/mlbookcamp-pants"}, None)

> Attenzione: i `float32` di numpy non sono serializzabili in JSON → convertire con `.tolist()` / `float()`.

## 5. Preparare l'immagine Docker

- Lambda supporta **container image**, utile per superare i limiti dei pacchetti zip.
- Si parte da un'**immagine base ufficiale AWS** per Lambda (es. `public.ecr.aws/lambda/python:3.x`).
- `tflite_runtime` si installa con un wheel precompilato adatto a Lambda (Alexey Grigorev mantiene wheel dedicati nel repo `tflite-aws-lambda`; in alternativa, versioni recenti disponibili via pip).
- `CMD` indica l'handler nel formato `file.funzione`.

In [ ]:
%%writefile Dockerfile
FROM public.ecr.aws/lambda/python:3.10

RUN pip install pillow numpy
RUN pip install --no-cache-dir tflite-runtime   # oppure il wheel specifico per Lambda

COPY clothing-model.tflite .
COPY lambda_function.py .

CMD ["lambda_function.lambda_handler"]

**Build ed esecuzione in locale:**

In [ ]:
# Nel terminale:
# docker build -t clothing-model .
# docker run -it --rm -p 8080:8080 clothing-model:latest

# Test dell'endpoint locale (Lambda Runtime Interface Emulator):
import requests
url = "http://localhost:8080/2015-03-31/functions/function/invocations"
data = {"url": "http://bit.ly/mlbookcamp-pants"}
print(requests.post(url, json=data).json())

## 6. Creare la funzione Lambda

Passaggi per il deploy dell'immagine:
1. Installare e configurare **AWS CLI** (`aws configure`).
2. Creare un repository **ECR** (Elastic Container Registry).
3. Fare login a ECR con Docker, **taggare** e fare **push** dell'immagine.
4. Dalla console Lambda: *Create function → Container image* e selezionare l'immagine su ECR.
5. **Aumentare memoria e timeout** (default troppo bassi: ad es. 1024+ MB e 30 s) perché il caricamento del modello è oneroso.
6. Testare con un *test event* `{"url": "..."}`.

In [ ]:
# Nel terminale (sostituire REGION e ACCOUNT_ID):
# aws ecr create-repository --repository-name clothing-tflite-images
#
# aws ecr get-login-password --region REGION | \
#   docker login --username AWS --password-stdin ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com
#
# REMOTE_URI=ACCOUNT_ID.dkr.ecr.REGION.amazonaws.com/clothing-tflite-images:clothing-model-001
# docker tag clothing-model:latest ${REMOTE_URI}
# docker push ${REMOTE_URI}

- La **prima invocazione** è lenta (*cold start*: caricamento del container e del modello); le successive sono rapide finché l'istanza resta "calda".
- Il costo dipende da memoria allocata × durata di esecuzione.

## 7. API Gateway: esporre la funzione Lambda

- Una Lambda da sola non è raggiungibile via HTTP pubblico: serve **Amazon API Gateway**.
- Passaggi nella console:
  1. Creare una **REST API** (nuova API)
  2. Creare una **risorsa** (es. `/predict`)
  3. Aggiungere un **metodo `POST`** con integrazione verso la funzione Lambda
  4. Testare il metodo dalla console
  5. **Deploy API** in uno *stage* (es. `test`) → si ottiene l'URL pubblico
- Il corpo della richiesta POST arriva alla Lambda; a seconda del tipo di integrazione (proxy o no) `event` può contenere direttamente il JSON o una stringa in `event["body"]`.

In [ ]:
import requests

url = "https://<API_ID>.execute-api.<REGION>.amazonaws.com/test/predict"
data = {"url": "http://bit.ly/mlbookcamp-pants"}

result = requests.post(url, json=data).json()
print(result)

> ⚠️ **Pulizia delle risorse**: al termine eliminare API Gateway, funzione Lambda e repository/immagini ECR per evitare costi inattesi.

---
## Riepilogo veloce

| Fase | Strumento | Cosa si fa |
|---|---|---|
| Modello | TF Lite | Conversione Keras → `.tflite`, inferenza leggera |
| Codice | `lambda_function.py` | `lambda_handler(event, context)`, modello caricato fuori dall'handler |
| Packaging | Docker | Immagine base AWS Lambda + `tflite_runtime` |
| Registry | ECR | Push dell'immagine |
| Esecuzione | AWS Lambda | Funzione da container, più memoria e timeout |
| Esposizione | API Gateway | REST API, risorsa `/predict`, metodo POST, deploy |